## Maintaining Agent State with ClaudeSDKClient

# Lesson 5: Conversational Workflows, Telemetry, and Stateful Sessions via `ClaudeSDKClient`

In the previous lesson, you unlocked the power of **Tools**, transforming your agent from a text-based conversationalist into an active assistant that can query networks, execute terminal tasks, and patch source documents.

However, all your interactions so far have relied on the atomic `query()` function. While `query()` is ideal for standalone automation tasks, it is completely **stateless**—the execution environment is destroyed at the end of each call, leaving the agent with zero memory of previous interactions.

In this lesson, you will learn how to transition to multi-turn conversational workflows using `ClaudeSDKClient`. This class maintains a persistent, stateful connection across multiple interaction loops, allowing you to build highly complex systems where the agent remembers past inputs, refines previous code patches, and responds to natural follow-up instructions.

---

## Stateful vs. Stateless Architectures

Understanding when to leverage the stateless `query()` function versus the stateful `ClaudeSDKClient` is essential for building resource-efficient, secure AI applications.

| Feature Execution Matrix | `query()` | `ClaudeSDKClient` |
| --- | --- | --- |
| **Session Persistence** | **Stateless.** Creates a brand-new subprocess instance for each invocation; context is discarded immediately upon resolution. | **Stateful.** Maintains a persistent local execution session across multiple conversation boundaries. |
| **Context Retention** | No historical memory between sequential calls. Every prompt starts from a completely clean slate. | Full conversation retention. The agent retains historical message context, code modifications, and tool results. |
| **Primary Use Cases** | One-off operations: standalone file parsing, automated test generation, or quick repository script adjustments. | Multi-turn interactions: complex interactive troubleshooting chat engines, multi-layered workflows, and custom dashboards. |
| **Lifecycle Hooks** | **Automated.** Subprocess instantiation, tool registration, execution, and cleanup are managed invisibly per call. | **Explicit.** Requires manual network bindings or structural Python asynchronous context managers (`async with`). |
| **Advanced System Features** | Supports built-in tools (`Read`, `Write`, `Bash`), but cannot register runtime lifecycle overrides or custom handlers. | Full support for user-defined custom tools, event intercept hooks, intermediate execution interrupts, and live confirmation callbacks. |

---

## Instantiating a `ClaudeSDKClient`

Instantiating a `ClaudeSDKClient` utilizes the exact same `ClaudeAgentOptions` configuration schema you mastered in earlier lessons. You define the underlying model target, inject specialized engineer system prompts, toggle active tool permissions, and map filesystem boundaries before passing the bundle to the client constructor:

```python
from pathlib import Path
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions

# 1. Package the structural execution constraints
options = ClaudeAgentOptions(
    model="haiku",
    system_prompt="You are a senior software architect specializing in DevOps pipelines.",
    max_turns=10,
    allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
    permission_mode="acceptEdits",
    cwd=Path(__file__).parent
)

# 2. Bind the options matrix to a stateful client instance
# At this stage, the client is initialized but its background subprocess is not yet active.
client = ClaudeSDKClient(options=options)

```

---

## Managing Session Lifecycles

Before you can dispatch messages through a stateful client, you must explicitly spin up its execution session. The SDK offers two distinct patterns for managing this lifecycle.

### Pattern A: Manual Connection Loops (For Custom Implementations)

This approach gives you absolute manual control over when the background process connects and disconnects, which is highly valuable when building persistent backend microservices:

```python
async def run_server_client():
    client = ClaudeSDKClient(options=options)
    
    # Explicitly spawn the background agent subprocess engine
    await client.connect()
    
    try:
        # Dispatch your conversational interactions here...
        pass
    finally:
        # Prevent memory leaks by guaranteeing resource teardown
        await client.disconnect()

```

### Pattern B: Asynchronous Context Managers (Recommended)

For standard workflows, Python's `async with` context block is the recommended approach. It guarantees clean session teardown and releases system resources automatically, even if an unhandled exception triggers a mid-turn crash:

```python
async def run_clean_context():
    # Automatically triggers connect() on entry and disconnect() on exit
    async with ClaudeSDKClient(options=options) as client:
        # Active session block space
        pass

```

---

## The Send-Receive Interaction Pattern

Unlike the stateless `query()` function—which returns an asynchronous iterator directly—the stateful `ClaudeSDKClient` decouples **sending an instruction** from **receiving the output stream**. This architectural separation gives you fine-grained control over streaming data pipelines and asynchronous message orchestration.

```python
async with ClaudeSDKClient(options=options) as client:
    # Step 1: Dispatch the objective string into the session pipeline
    await client.query("Search the web for the current stable version of the Claude Agent SDK.")
    
    # Step 2: Open an independent stream listener to harvest the live execution frames
    async for message in client.receive_response():
        # Yields structural message objects identically to the query() function loop
        pass

```

### Building a Reusable Output Stream Multiplexer

To keep multi-turn code bases clean and highly maintainable, encapsulate your `receive_response()` filtering loops into a single shared helper routine:

```python
from claude_agent_sdk import ClaudeSDKClient, AssistantMessage, TextBlock, ToolUseBlock

async def display_live_stream(client: ClaudeSDKClient):
    """Asynchronously harvests and prints live agent turns from the active session"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:\n{block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Invoking Engine Tool]: {block.name}")

```

---

## Conducting a Multi-Turn Conversational Session

The real power of `ClaudeSDKClient` becomes apparent when executing sequential instructions where subsequent steps explicitly depend on earlier results. Because the conversation history is preserved within the session context, the agent can understand relative contextual terms like *"the documentation you just pulled."*

```python
import anyio

async def run_architect_workflow():
    async with ClaudeSDKClient(options=options) as client:
        
        # =====================================================================
        # TURN 1: Gathering Technical Specifications
        # =====================================================================
        print("=== Dispatching Turn 1 ===")
        await client.query(
            "Search the web for the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        await display_live_stream(client)
        
        print("\n" + "="*50 + "\n")
        
        # =====================================================================
        # TURN 2: Building Upon Retained Context
        # =====================================================================
        print("=== Dispatching Turn 2 ===")
        # This relative reference is perfectly understood because of session memory!
        await client.query(
            "Now edit the file ./summaries/architecture.md and append a new section "
            "explaining the core tradeoffs you just discovered in the step above."
        )
        await display_live_stream(client)

if __name__ == "__main__":
    anyio.run(run_architect_workflow)

```

---

## Tracking the Multi-Turn Execution Stream

When you execute this multi-turn script, you will see the logs unfold sequentially. Notice how information gathered in the first transaction directly drives the behavior of the second step:

### Output Trace: Turn 1 (Information Gathering)

```text
=== Dispatching Turn 1 ===
🔧 [Invoking Engine Tool]: WebSearch

💬 Claude Response:
I have performed a search across the developer networks. In summary, `query()` provides an atomic, stateless, one-shot interface with zero memory preservation between calls. Conversely, `ClaudeSDKClient` provisions a full, stateful runtime container capable of tracking conversation history and handling advanced process interrupts.

```

### Output Trace: Turn 2 (Contextual Execution)

```text
=== Dispatching Turn 2 ===
🔧 [Invoking Engine Tool]: Read
🔧 [Invoking Engine Tool]: Edit

💬 Claude Response:
Perfect! I have parsed your target file layout via `Read`, and applied localized patches using the `Edit` tool. I appended a structured technical breakdown mapping the stateless vs stateful performance trade-offs we discovered during our previous search, aligning the new blocks cleanly with your project rules.

```

If you had attempted to execute this identical chain using two back-to-back stateless `query()` calls, the second step would have failed immediately with a missing context error because the second instance would have no knowledge of what *"the trade-offs discovered in the step above"* referred to.

---

## Handling Mid-Turn Process Interrupts

In high-throughput automation environments, you will often need to halt an active agent run immediately—whether because a tool execution has hung, a process has veered off course, or a user has manually triggered a cancellation event. `ClaudeSDKClient` supports graceful session interrupts using the `.interrupt()` method.

To safely fire an interrupt while simultaneously reading an active generation stream, combine your execution blocks using an asynchronous task group (`anyio.create_task_group`):

```python
import anyio

async def run_interruptible_session():
    async with ClaudeSDKClient(options=options) as client:
        # 1. Trigger a heavy, multi-layered operations prompt
        await client.query("Search the web for deep architecture benchmarks across every Claude version.")
        
        # 2. Declare a concurrent timeout monitor function
        async def trigger_safeguard_kill():
            # Allow the agent to work for a fixed duration
            await anyio.sleep(3)
            print("\n⚠️ [Safety Monitor Triggered]: Interrupting active agent session process...")
            # Gracefully halt the current reasoning/tool loop execution turn
            await client.interrupt()
            
        # 3. Multiplex both streams concurrently inside a managed task group
        async with anyio.create_task_group() as tg:
            tg.start_soon(display_live_stream, client) # Read output streams
            tg.start_soon(trigger_safeguard_kill)     # Listen for timeout boundaries
            
        # =====================================================================
        # POST-INTERRUPT RECOVERY LAYER
        # =====================================================================
        print("\n" + "="*60)
        print("Pipeline Recovered. Dispatching Simplified Fallback Command.")
        print("="*60)
        
        # The background session remains fully functional even after an interrupt!
        await client.query("Instead of a web search, tell me in one short sentence what ClaudeSDKClient is.")
        await display_live_stream(client)

```

### The Interrupt Lifecycle Decoded:

1. **The Interrupt Command:** Calling `await client.interrupt()` signals the underlying local engine subprocess to halt its current task loop immediately.
2. **Graceful Error Catching:** The active stream generator stops safely and yields a final `ResultMessage` containing `subtype='error_during_execution'`, allowing you to handle the termination gracefully.
3. **Session Continuity:** Crucially, the master session wrapper remains fully intact. You do not need to rebuild the client or reload your workspace configurations; you can immediately issue a brand-new `client.query()` down the same pipe.

---

## Summary

You have expanded your agent coordination capabilities by mastering persistent, stateful application logic:

* **`ClaudeSDKClient`** preserves session memory, files tokens, and keeps tracking context alive across consecutive communication exchanges.
* **Asynchronous Context Managers (`async with`)** automate background connection lifecycles, guaranteeing safe system cleanup and eliminating resource leaks.
* **Decoupled Messaging Methods (`query()` + `receive_response()`)** allow you to implement flexible concurrency and custom stream parsing architectures.
* **The `.interrupt()` Controller API** provides complete runtime sovereignty, giving you the ability to stop running background processes instantly without breaking your application state.

## From Stateless Queries to Session Clients

Now it's time to put the lesson concepts into practice by transforming a stateless query into a session-based conversation. The starter code uses the query() function to send a single request, but you'll convert it to use ClaudeSDKClient instead, which will set the foundation for multi-turn conversations in upcoming exercises.

Your task is to replace the stateless query() call with the client-based pattern. Here's what you need to do:

    Create a ClaudeSDKClient using the existing options and wrap it in an async with context manager to handle the session lifecycle.
    Send the query using await client.query() with just the prompt string (the options are already in the client).
    Receive the response using client.receive_response() in your async for loop instead of calling query() directly.

The message handling logic stays exactly the same — you're only changing how you send and receive messages. Once you complete this transformation, you'll have a working session-based client ready for multi-turn conversations!

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeAgentOptions,
    query,
    AssistantMessage,
    TextBlock,
    ClaudeSDKClient,
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # TODO: Replace the stateless query() call with a session-based client pattern
    # - Create a ClaudeSDKClient using the options with an async context manager
    # - Send the query using client.query() (pass only the prompt, not the options)
    # - Receive the response using client.receive_response() in an async for loop
    async for message in query(
        prompt=(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        ),
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)


if __name__ == "__main__":
    anyio.run(main)
```

Here is the updated code converted to use the stateful `ClaudeSDKClient` within an `async with` context manager.

This structure decouples sending the instruction from harvesting the streaming generator, setting up your workspace for multi-turn conversations:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeAgentOptions,
    query,
    AssistantMessage,
    TextBlock,
    ClaudeSDKClient,
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Replace the stateless query() call with a session-based client pattern
    # - Create a ClaudeSDKClient using the options with an async context manager
    async with ClaudeSDKClient(options=options) as client:
        # - Send the query using client.query() (pass only the prompt, not the options)
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        
        # - Receive the response using client.receive_response() in an async for loop
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(block.text)


if __name__ == "__main__":
    anyio.run(main)

```

### Architectural Key Points:

* **`async with` Automation:** Entering the block spins up the background subprocess and authenticates with the remote cloud model. Exiting the block ensures that resources are torn down cleanly.
* **Separation of Concerns:** `await client.query()` handles the task dispatch, while `client.receive_response()` monitors the socket for upcoming tokens and tool updates.

## Building a Reusable Response Display Helper

You have probably noticed that the message handling code (the async for loop with all the isinstance checks and print statements) takes up quite a bit of space in your main function. As you build more complex multi-turn conversations, you will need to receive and display responses multiple times, and copying this code over and over would make your program messy and hard to maintain.

The solution is to extract this logic into a reusable helper function. Your task is to create a function that encapsulates the message receiving and display pattern, then use it to clean up your main function. Here is what you need to do:

    Create an async function that takes a ClaudeSDKClient as its parameter (with the proper type hint).
    Move the entire message receiving and display logic into this helper function — the async for loop that iterates over client.receive_response(), the isinstance checks for AssistantMessage, and the nested loop handling TextBlock and ToolUseBlock content.
    Replace the inline message handling code in your main function with a simple call to your helper function.

Do not forget to add a docstring to your helper function explaining what it does! Once you complete this refactoring, your code will be much cleaner and ready for multi-turn conversations where you will call this helper multiple times.

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


# TODO: Define an async helper function that takes a ClaudeSDKClient parameter
    # TODO: Move the async for loop that iterates over client.receive_response() into this function
        # TODO: Move the isinstance and the nested loop for TextBlock and ToolUseBlock into this function


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Use ClaudeSDKClient for stateful conversation
    async with ClaudeSDKClient(options=options) as client:
        # Send the query to the agent
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        
        # TODO: Replace this inline message handling with a call to your helper function
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"\n💬 Claude Response:")
                        print(block.text)
                    elif isinstance(block, ToolUseBlock):
                        print(f"\n🔧 [Tool: {block.name}]")


if __name__ == "__main__":
    anyio.run(main)

```

Here is the complete refactored code with the streaming logic cleanly extracted into a reusable asynchronous helper function.

Creating a dedicated output handler significantly reduces boilerplate code and prepares your script for multi-turn interactions where you will need to display responses repeatedly.

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


# Define an async helper function that takes a ClaudeSDKClient parameter
async def display_response(client: ClaudeSDKClient):
    """
    Asynchronously reads the stream from the ClaudeSDKClient and prints
    formatted text responses and tool execution blocks.
    """
    # Move the async for loop that iterates over client.receive_response() into this function
    async for message in client.receive_response():
        # Move the isinstance and the nested loop for TextBlock and ToolUseBlock into this function
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Use ClaudeSDKClient for stateful conversation
    async with ClaudeSDKClient(options=options) as client:
        # Send the query to the agent
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        
        # Replace this inline message handling with a call to your helper function
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Technical Advantages of this Refactor:

* **Code Cleanliness:** Your `main()` workflow loop is now incredibly readable. It models a clean structural design pattern: configure options $\rightarrow$ initialize client session $\rightarrow$ query task objective $\rightarrow$ display stream outcomes.
* **Stream Multiplexing Readiness:** Because the helper functions ingest the generic `ClaudeSDKClient` interface type hint, you can easily scale or extend this helper later to also trace `UserMessage` types or capture performance metadata (`ResultMessage`) globally without dirtying your main functional steps.

## Verifying Session State with Follow-Up Queries

Excellent work building that reusable helper function — your code is now clean and ready for the real power of ClaudeSDKClient to shine! Now it's time to demonstrate why session-based conversations matter by adding a second query that proves the agent remembers previous interactions.

Your task is to send a follow-up message that builds on the first conversation. The first query already asks the agent to search for and explain the difference between query() and ClaudeSDKClient, and that information becomes part of the session's memory. Now, you need to add a second query that references this information using contextual phrases like "the difference you just found" or "the information you discovered."

Here's what you need to do:

    Send a follow-up query using await client.query() that asks the agent to edit the file ./summaries/claude-agent-sdk.md and add a new section explaining the difference it found in the first conversation.
    Display the response using your display_response() helper function.

The key is that your second query should use contextual references that only make sense if the agent remembers the first conversation — this is what proves the session maintains state! Once you complete this, you'll see the agent successfully use information from the first query to complete the second task, demonstrating the true power of conversational agents.


```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Use ClaudeSDKClient for stateful conversation with context
    async with ClaudeSDKClient(options=options) as client:
        
        # First query: Search for information
        print("=" * 60)
        print("First Message: Search for differences")
        print("=" * 60)
        
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        await display_response(client)
        
        # Second query: Edit the file (context is maintained from first query)
        print("\n" + "=" * 60)
        print("Second Message: Edit the summary file")
        print("=" * 60)
        
        # TODO: Send a follow-up query that references the information from the first conversation
        
        # TODO: Display the response using the helper function


if __name__ == "__main__":
    anyio.run(main)
```

Here is the completed code with your follow-up query added.

By using relative phrasing like *"the difference you just found"*, you are forcing the agent to rely on its session memory to complete the task. This is the exact moment a basic script transforms into a stateful, context-aware workflow!

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Use ClaudeSDKClient for stateful conversation with context
    async with ClaudeSDKClient(options=options) as client:
        
        # First query: Search for information
        print("=" * 60)
        print("First Message: Search for differences")
        print("=" * 60)
        
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        await display_response(client)
        
        # Second query: Edit the file (context is maintained from first query)
        print("\n" + "=" * 60)
        print("Second Message: Edit the summary file")
        print("=" * 60)
        
        # Send a follow-up query that references the information from the first conversation
        await client.query(
            "Now edit the file ./summaries/claude-agent-sdk.md and add a new section "
            "at the end explaining the difference you just found."
        )
        
        # Display the response using the helper function
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### What You Will See in the Output:

When you run this script, pay close attention to Turn 2. The agent won't execute a second web search. Instead, you'll see it immediately use the `Read` tool to scan your local markdown file, followed by the `Edit` tool to patch the file using the exact technical differences it gathered during Turn 1.

If this were a stateless `query()` loop, the second execution would have failed because the agent wouldn't know what "difference" you were talking about!

## Manual Session Lifecycle with Connect and Disconnect

Great progress on building that multi-turn conversation! Now, let's explore an alternative way to manage the session lifecycle that gives you more explicit control over when connections start and end.

The async with context manager you've been using is convenient because it automatically handles connection and cleanup, but sometimes you need more fine-grained control over the session lifecycle. The ClaudeSDKClient also supports manual connection management through explicit connect() and disconnect() calls, which can be useful when you need to control exactly when resources are allocated or when you're integrating with frameworks that don't work well with context managers.

Your task is to transform your existing code to use manual session management instead of the context manager pattern. Here's what you need to do:

    Remove the async with statement and create the client directly by assigning ClaudeSDKClient(options=options) to a variable.
    Add an explicit await client.connect() call to start the session.
    Wrap all your conversation code (both queries and their display calls) in a try-finally block to ensure proper cleanup, even if errors occur.
    Add an explicit await client.disconnect() call in the finally block to clean up the session.

The conversation logic itself stays exactly the same — you're only changing how you manage the connection lifecycle. Once you complete this transformation, you'll understand both patterns for session management and can choose the right one for your specific use case!


```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # TODO: Replace the async context manager with manual session management:
    # - Create the client directly (do not use 'async with')
    # - Manually connect to the client using 'await client.connect()'
    # - Wrap the conversation code in a try-finally block and call disconnect() in finally
    async with ClaudeSDKClient(options=options) as client:
        
        # First query: Search for information
        print("=" * 60)
        print("First Message: Search for differences")
        print("=" * 60)
        
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        await display_response(client)
        
        # Second query: Edit the file (context is maintained from first query)
        print("\n" + "=" * 60)
        print("Second Message: Edit the summary file")
        print("=" * 60)
        
        await client.query(
            "Now edit the file ./summaries/claude-agent-sdk.md and add a new section "
            "at the end explaining the difference you just found."
        )
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)
```

Here is the updated code converted to use manual session management with an explicit `try-finally` execution pattern.

This gives your application direct control over the exact lifecycle bounds of the agent's background process:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Replace the async context manager with manual session management:
    # - Create the client directly (do not use 'async with')
    client = ClaudeSDKClient(options=options)
    
    # - Manually connect to the client using 'await client.connect()'
    await client.connect()
    
    # - Wrap the conversation code in a try-finally block and call disconnect() in finally
    try:
        # First query: Search for information
        print("=" * 60)
        print("First Message: Search for differences")
        print("=" * 60)
        
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        await display_response(client)
        
        # Second query: Edit the file (context is maintained from first query)
        print("\n" + "=" * 60)
        print("Second Message: Edit the summary file")
        print("=" * 60)
        
        await client.query(
            "Now edit the file ./summaries/claude-agent-sdk.md and add a new section "
            "at the end explaining the difference you just found."
        )
        await display_response(client)
        
    finally:
        # Guarantee that resource cleanup runs regardless of execution errors
        await client.disconnect()


if __name__ == "__main__":
    anyio.run(main)

```

### Why the `try-finally` Block is Vital:

When managing connections manually, if a runtime error or network timeout occurs during `await client.query()`, standard sequential execution breaks. By nesting your active session logic inside the `try` block and placing `await client.disconnect()` inside the `finally` block, Python guarantees that the underlying agent engine subprocess closes down cleanly—protecting your system from rogue zombie processes and resource leaks!

## Interrupting Agents for Better Control

Excellent work demonstrating how session state enables contextual conversations! Now, let's explore a powerful control mechanism that becomes essential when building interactive applications — the ability to interrupt an agent mid-execution when things aren't going as planned.

Sometimes, you need to stop an agent while it's working — maybe it's taking too long, heading in the wrong direction, or you realize you asked the wrong question. The ClaudeSDKClient supports interrupts through the interrupt() method, which allows you to cancel an ongoing agent run and then continue the conversation with a different request.

Your task is to transform the existing single-query code into an interrupt demonstration. Here's what you need to do:

    Create a nested async function called cancel_after_delay() that waits 3 seconds using anyio.sleep(3), prints an interruption warning message, and calls await client.interrupt().
    Replace the simple display_response(client) call with a task group pattern using async with anyio.create_task_group() as tg: that runs both display_response(client) and cancel_after_delay() concurrently with tg.start_soon().
    After the task group completes, add separator print statements and send a simpler follow-up query asking for just a one-sentence explanation of what ClaudeSDKClient is used for.
    Display the follow-up response using your helper function.

The key insight is that interrupts don't end the session — they just stop the current request, allowing you to continue the conversation with a new query. Once you complete this exercise, you'll understand how to build responsive applications that give users control over long-running agent operations!

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Use ClaudeSDKClient for stateful conversation with context
    async with ClaudeSDKClient(options=options) as client:
        
        # Start a query that we'll interrupt
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        
        # TODO: Create a nested async function called cancel_after_delay() that:
        # - Waits 3 seconds using anyio.sleep(3)
        # - Prints "\n⚠️ Interrupting agent..."
        # - Calls await client.interrupt()
        
        # TODO: Replace the simple display_response call below with a task group pattern:
        # - Use async with anyio.create_task_group() as tg:
        # - Run display_response(client) and cancel_after_delay() concurrently using tg.start_soon()
        await display_response(client)
        
        # TODO: After the task group, add a follow-up query section:
        # - Print separator lines (60 equals signs) with a message about the new request
        # - Send a simpler query asking for a one-sentence explanation of what ClaudeSDKClient is used for
        # - Display the response using the helper function


if __name__ == "__main__":
    anyio.run(main)
```

Here is the completed code transformed into a fully operational async task group pattern to showcase execution interruptions.

By running your output stream concurrently alongside a delayed cancellation safeguard monitor via `anyio.create_task_group()`, you can forcefully halt active engine cycles without breaking the master stateful session wrapper:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def display_response(client: ClaudeSDKClient):
    """Helper function to display agent messages with formatting"""
    async for message in client.receive_response():
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are a helpful assistant.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write", "Edit"],
        permission_mode="acceptEdits",
        cwd=Path(__file__).parent
    )
    
    # Use ClaudeSDKClient for stateful conversation with context
    async with ClaudeSDKClient(options=options) as client:
        
        # Start a query that we'll interrupt
        await client.query(
            "Search the web for what's the difference between using query() and "
            "ClaudeSDKClient in the Claude Agent SDK. Give me a brief explanation."
        )
        
        # Create a nested async function called cancel_after_delay()
        async def cancel_after_delay():
            # Waits 3 seconds using anyio.sleep(3)
            await anyio.sleep(3)
            # Prints an interruption warning message
            print("\n⚠️ Interrupting agent...")
            # Calls await client.interrupt()
            await client.interrupt()
        
        # Replace the simple display_response call below with a task group pattern:
        # - Use async with anyio.create_task_group() as tg:
        async with anyio.create_task_group() as tg:
            # - Run display_response(client) and cancel_after_delay() concurrently using tg.start_soon()
            tg.start_soon(display_response, client)
            tg.start_soon(cancel_after_delay)
        
        # After the task group, add a follow-up query section:
        # - Print separator lines (60 equals signs) with a message about the new request
        print("\n" + "=" * 60)
        print("New Message: Simplified request after interrupt")
        print("=" * 60)
        
        # - Send a simpler query asking for a one-sentence explanation of what ClaudeSDKClient is used for
        await client.query(
            "Instead, just tell me in one sentence what ClaudeSDKClient is used for."
        )
        
        # - Display the response using the helper function
        await display_response(client)


if __name__ == "__main__":
    anyio.run(main)

```

### Key Asynchronous Mechanics:

* **The `anyio` Task Group Pattern:** Structuring code using `async with anyio.create_task_group()` creates a non-blocking playground where multiple asynchronous routines execute concurrently. The main program execution will wait at the exit boundary of the context manager block until all registered background worker tasks resolve or handle termination.
* **Graceful Recovery Loop:** When `client.interrupt()` hits the active subprocess stream, it safely shuts down the current execution cycle. Crucially, the master session context is preserved—allowing your fallback query to be processed immediately under the same connection footprint!